# Teesri Shikayat — narration with Chatterbox

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.** If the second cell stops with an error, use **Runtime → Restart session**, then run the second cell again. It installs Chatterbox (about 3 minutes), then reads every sentence once (take a), then again (take b), about 30 minutes in all.
3. Every 11 sentences your browser downloads a zip (`vo-a-01-11.zip` …). Leave them in your Downloads folder. If the runtime drops, rerun the second cell: on the same runtime it skips the takes already made.

Settings are copied from Beacon's public video/narrate.py: Chatterbox's built-in voice (no reference recording), exaggeration=0.4, cfg_weight=0.45, temperature=0.7.

The sentences come from `video/narration/lines.json` (every variant: agent mode, template mode, the ALT line), made by `scripts/narration_lines.py`.

In [ ]:
!pip -q install chatterbox-tts==0.1.7 && pip -q uninstall -y torchvision  # Colab's torchvision is built for a newer torch and breaks transformers' import

In [ ]:
import json, os, zipfile, torch, torchaudio as ta
from importlib.metadata import version
from chatterbox.tts import ChatterboxTTS
from google.colab import files
LINES = json.loads(r'''[{"n": 1, "say": "Every monsoon in Mumbai, the water from my tap turns dirty."}, {"n": 2, "say": "I never knew if the house next door had it too."}, {"n": 3, "say": "In Indore last December, a leaking pipe pulled sewage into the taps."}, {"n": 4, "say": "People complained one by one."}, {"n": 5, "say": "Thirty-six died."}, {"n": 6, "say": "This is Teesri Shikayat, the third complaint."}, {"n": 7, "say": "When three homes close together report dirty water, everyone around them is warned."}, {"n": 8, "say": "And only the residents can close the case."}, {"n": 9, "say": "Mumbai logged over fifteen hundred dirty-water complaints this year."}, {"n": 10, "say": "Last week, the city said: warn residents at once."}, {"n": 11, "say": "Nothing new to learn."}, {"n": 12, "say": "Scan a code, tap Start on Telegram, share your location, and you're in."}, {"n": 13, "say": "Then you complain the way people actually do: a voice note, in Hindi."}, {"n": 14, "say": "Amazon Transcribe writes it down, and Nova pulls out what matters: the colour, the smell, how long, and who's sick."}, {"n": 15, "say": "Two neighbours already reported."}, {"n": 16, "say": "Mine is the third within two hundred and fifty metres in three days."}, {"n": 17, "say": "The tripwire fires, and the ring is marked as a likely pipe leak."}, {"n": 18, "say": "A case agent built with Strands reads all three complaints and briefs a local volunteer in Hindi."}, {"n": 19, "say": "One tap to approve."}, {"n": 20, "say": "Everyone in the ring hears this."}, {"n": 21, "say": "Twenty of these homes never complained."}, {"n": 22, "say": "The agent writes the ward office a formal complaint."}, {"n": 23, "say": "Names and numbers never leave the lane."}, {"n": 24, "say": "Cedar enforces that."}, {"n": 25, "say": "Two days later, the ward office replies: resolved."}, {"n": 26, "say": "The agent believes them and tries to close the case."}, {"n": 27, "say": "Cedar says no."}, {"n": 28, "say": "Only residents can close it."}, {"n": 29, "say": "So it asks them."}, {"n": 30, "say": "My tap says no."}, {"n": 31, "say": "The case reopens."}, {"n": 32, "say": "Replayed on Indore's dates, the third complaint would have raised the alarm about ten days before the first person fell ill."}, {"n": 33, "say": "It all runs serverless on AWS."}, {"n": 34, "say": "Step Functions holds each case for days, DynamoDB streams feed the tripwire, and Polly speaks Hindi."}, {"n": 35, "say": "One incident costs about one rupee."}, {"n": 36, "say": "Built for the monsoon that floods the street and dirties the tap."}, {"n": 37, "say": "Teesri Shikayat: closed at the tap, not on paper."}, {"n": 38, "say": "Amazon Transcribe writes it down, and three quick taps confirm what matters: the colour, the smell, and how long."}, {"n": 39, "say": "A case opens in Step Functions."}, {"n": 40, "say": "It sums up all three complaints for a local volunteer, in Hindi."}, {"n": 41, "say": "The ward office gets a formal complaint."}, {"n": 42, "say": "Cedar checks that."}, {"n": 43, "say": "That's a request to close the case, and Cedar says no."}, {"n": 44, "say": "Dozens died."}]''')
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU, then Run all again"
model = ChatterboxTTS.from_pretrained(device="cuda")
os.makedirs("vo", exist_ok=True)
manifest = {"chatterbox": version("chatterbox-tts"), "torch": torch.__version__, "sr": model.sr,
            "settings": {"exaggeration": 0.4, "cfg_weight": 0.45, "temperature": 0.7}, "takes": []}
BATCH = 11  # a zip every 11 sentences, so a dropped runtime loses at most one batch
for take, base in (("a", 0), ("b", 1000)):  # every take a first: a late drop still leaves one take of each sentence
    for i in range(0, len(LINES), BATCH):
        part = LINES[i:i + BATCH]
        for l in part:
            path = f"vo/{l['n']:02d}{take}.wav"
            if os.path.exists(path):  # rerun on the same runtime: keep what is made
                wav, _ = ta.load(path)
            else:
                torch.manual_seed(base + l["n"])
                wav = model.generate(l["say"], exaggeration=0.4, cfg_weight=0.45, temperature=0.7)
                ta.save(path, wav, model.sr)
            secs = wav.shape[-1] / model.sr
            manifest["takes"].append({"n": l["n"], "take": take, "seed": base + l["n"], "seconds": round(secs, 2), "chars": len(l["say"])})
            print(f"{l['n']:02d}{take} {secs:5.1f}s  {l['say'][:60]}", flush=True)
        json.dump(manifest, open("vo/manifest.json", "w"), indent=1)
        name = f"vo-{take}-{part[0]['n']:02d}-{part[-1]['n']:02d}.zip"
        with zipfile.ZipFile(name, "w") as z:
            for l in part: z.write(f"vo/{l['n']:02d}{take}.wav")
            z.write("vo/manifest.json")
        files.download(name)
        print("downloaded", name, flush=True)